# JEPA Weight Sweep — Deep Validation

**Why**: JEPA (cross-tick latent prediction) gives +9pp on visual tasks at w=0.1. Higher weights cause instability on cifar10.

**Goal**: Fine-grained weight sweep (0.05 / 0.1 / 0.2) with 3 seeds on visual tasks to find the optimal operating point.

**Hardware**: 1 machine x 8 GPUs. 18 runs, ~6h.

Run from the `paper/` directory.

In [ ]:
import sys; sys.path.insert(0, '.'); sys.path.insert(0, '..')
from exp_runner import make_jepa, run_all, status, collect, plot_delta_bars
%matplotlib inline

In [ ]:
TASKS = ['cifar10', 'mazes']
SEEDS = [0, 1, 2]
WEIGHTS = [0.05, 0.1, 0.2]

exps = make_jepa(tasks=TASKS, seeds=SEEDS, weights=WEIGHTS)
print(f'{len(exps)} experiments')
for e in exps[:6]:
    print(f'  {e.name}')
print('  ...')

## Step 1 — Dry run

In [ ]:
run_all(exps, gpus=8, log_root='logs/deep/02_jepa', dry_run=True)

## Step 2 — Run training

Uncomment to launch (~6h).

In [ ]:
# done, failed = run_all(exps, gpus=8, log_root='logs/deep/02_jepa')

In [ ]:
status('logs/deep/02_jepa')

## Step 3 — Results

Plot weight sweep curves per task.

In [ ]:
df = collect('logs/deep/02_jepa')
if df.empty:
    print('No results yet.')
else:
    print(df[['name', 'task', 'best_acc', 'delta']].to_string(index=False))
    plot_delta_bars(df, 'JEPA weight sweep vs baseline', 'figures/02_jepa_delta.png')

In [ ]:
# Weight sweep curves per task
if not df.empty and 'best_acc' in df and df['best_acc'].notna().any():
    import matplotlib.pyplot as plt
    import re, numpy as np
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for ax, task in zip(axes, ['cifar10', 'mazes']):
        sub = df[df['task'] == task].copy()
        sub['w'] = sub['name'].str.extract(r'w([0-9]+p?[0-9]*)').iloc[:, 0].str.replace('p', '.').astype(float)
        for w in sorted(sub['w'].unique()):
            s = sub[sub['w'] == w]
            ax.errorbar([w], [s['best_acc'].mean() * 100],
                        yerr=[s['best_acc'].std(ddof=1) * 100 if len(s) > 1 else 0],
                        fmt='o', capsize=5, markersize=9, color='#1f77b4' if task == 'cifar10' else '#ff7f0e')
        from exp_runner import BASELINE_ACC
        ax.axhline(BASELINE_ACC[task] * 100, color='gray', ls='--', alpha=0.5, label='baseline')
        ax.set_xlabel('jepa_weight')
        ax.set_ylabel('best test acc (%)')
        ax.set_title(task)
        ax.legend()
    fig.suptitle('JEPA weight sweep (errorbar = std over 3 seeds)')
    fig.tight_layout()
    fig.savefig('figures/02_jepa_weight_curve.png', dpi=150, bbox_inches='tight')
    plt.show()